# Public API Examples

This notebook demonstrates the core `clifford` package API: configuring an algebra,
constructing multivectors, and computing inverses.

In [1]:
from clifford.context import Cl, Layout
from clifford.multivector import Accum
from clifford.inverse import fls_inverse, euclidean_inverse, sparse_fls_inverse

## Configuring an algebra

`Cl(p, q, r)` sets the global algebra to one with `p` positive, `q` negative, and `r` null
basis vectors.  `Layout([...])` is equivalent but takes a per-axis sign list.

In [2]:
Cl(6)          # Euclidean 6D: Cl(6, 0, 0)
print("Euclidean 6D configured")

Layout([1, 1, 1, -1, -1, -1])   # mixed signature, still 6D
print("Mixed 6D configured")

Cl(6)          # back to Euclidean for the rest of the notebook

Euclidean 6D configured
Mixed 6D configured


## Constructing multivectors

`Accum` holds a multivector as a NumPy array of $2^d$ real coefficients.
Coefficients are indexed by ordinal: bit $k$ set in the index means basis vector
$e_{k+1}$ is present in that blade.

In [3]:
A = Accum()
A.random()
print("Random 6D multivector (64 coefficients):")
print(A)

Random 6D multivector (64 coefficients):
00000000       1.14922220
00000001       0.83493434
00000010       1.16988656
00000011       2.94583196
00000100      -1.45148820
00000101       0.84521200
00000110      -0.31724493
00000111      -0.59700946
00001000      -0.36892028
00001001      -1.04432742
00001010       0.30666941
00001011       0.49486728
00001100       0.24547014
00001101      -0.29228796
00001110       0.07782393
00001111      -0.79512344
00010000       0.04639660
00010001      -1.14825127
00010010       1.39277963
00010011      -0.50571071
00010100      -0.98223851
00010101       0.88868168
00010110      -0.07656469
00010111      -0.30172120
00011000       0.95895994
00011001       1.01455050
00011010       1.34367301
00011011       0.32055233
00011100      -0.11072768
00011101      -0.12673208
00011110      -0.04461439
00011111       2.76900734
00100000       0.26867978
00100001       1.19492506
00100010      -0.52256440
00100011      -0.71605506
00100100      -2.338433

In [4]:
# Scalar, vector, and pseudoscalar components
import clifford.context as ctx

print(f"Scalar  (grade 0): A[0b000000] = {A.Reg[0]:.6f}")
print(f"e1      (grade 1): A[0b000001] = {A.Reg[1]:.6f}")
print(f"e1^e2   (grade 2): A[0b000011] = {A.Reg[3]:.6f}")
print(f"e1^..^e6 (grade 6, pseudoscalar): A[0b111111] = {A.Reg[63]:.6f}")

Scalar  (grade 0): A[0b000000] = 1.149222
e1      (grade 1): A[0b000001] = 0.834934
e1^e2   (grade 2): A[0b000011] = 2.945832
e1^..^e6 (grade 6, pseudoscalar): A[0b111111] = -1.668423


## Computing inverses

### FLS inverse (general multivector)

In [5]:
iA = fls_inverse(A)
check = A * iA
print("A * fls_inverse(A):")
print(check)
print(f"\nScalar component (should be 1.0): {check.Reg[0]:.15f}")
print(f"Max off-scalar residual:           {max(abs(check.Reg[1:])):.2e}")

A * fls_inverse(A):
00000000       1.00000000


Scalar component (should be 1.0): 1.000000000000000
Max off-scalar residual:           1.06e-15


### Euclidean inverse (Euclidean signature, dimensions 6–13)

In [6]:
iA2 = euclidean_inverse(A)
check2 = A * iA2
print(f"Scalar component: {check2.Reg[0]:.15f}")
print(f"Max off-scalar:   {max(abs(check2.Reg[1:])):.2e}")

Scalar component: 1.000000000000000
Max off-scalar:   1.01e-15


## Algebra arithmetic

`Accum` supports `*` (geometric product), `+`, `-`, and scalar multiplication.

In [7]:
B = Accum()
B.random()

# geometric product is non-commutative in general
AB = A * B
BA = B * A

commutator = AB - BA
print(f"||AB - BA||_max = {max(abs(commutator.Reg)):.6f}  (non-zero means non-commutative)")

||AB - BA||_max = 27.362306  (non-zero means non-commutative)


## Verify across dimensions

In [8]:
import numpy as np

for d in [6, 7, 8, 9, 10, 11, 12, 13]:
    Cl(d)
    A = Accum(); A.random()
    iA = euclidean_inverse(A)
    residual = max(abs((A * iA).Reg[1:]))
    print(f"Cl({d:2d}):  max residual = {residual:.2e}")

Cl( 6):  max residual = 9.50e-15
Cl( 7):  max residual = 8.43e-16
Cl( 8):  max residual = 3.67e-14
Cl( 9):  max residual = 4.87e-12
Cl(10):  max residual = 1.05e-10
Cl(11):  max residual = 1.59e-11
Cl(12):  max residual = 1.46e-15
Cl(13):  max residual = 1.20e-15
